[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/YOUR-GITHUB-USERNAME/JAXCode/blob/master/templates/b_27_cross_attention_pure.ipynb)

# 🟡 Medium: Cross-Attention without Flax

*Attention & Transformers*
Problem 23 with no Flax — and, if you have done `b_26`, almost no new code.

### Signature
```python
class MultiHeadCrossAttention:
    def __init__(self, d_model, num_heads, *, key): ...
    def __call__(self, x_q, x_kv): ...
    # (B, seq_q, d_model), (B, seq_kv, d_model) -> (B, seq_q, d_model)
```

Same four projections as `b_26`: `W_q`, `W_k`, `W_v`, `W_o`, each
`Linear(d_model, d_model)`, from `jax.random.split(key, 4)`.

### The one line that differs
```python
q = self.W_q(x_q)      # queries from one sequence
k = self.W_k(x_kv)     # keys and values from the other
v = self.W_v(x_kv)
```

That is genuinely all of it — which is why this sits right after `b_26`.

### The trap it adds
`seq_q` and `seq_kv` **differ**. The scores are `(..., H, seq_q, seq_kv)`, the
output length comes from `x_q`, and softmax runs over the last axis (the keys).
Anything that assumed a square score matrix breaks here, and a square test case
would not notice.

### A property worth checking yourself
Feed the same array as both inputs and you must get exactly self-attention
back. That single assertion catches most wiring mistakes — a swapped
`x_q`/`x_kv`, or `W_k` reading the wrong sequence.

### Why this exists alongside problem 23
Interview sandboxes often ship `jax` alone. The API is kept as close to the
`nnx` version as it can be — same class name, same arguments, same attribute
names — so practising it reinforces problem 23 instead of competing with it.
Only `rngs=nnx.Rngs(params=0)` becomes `key=jax.random.key(0)`, and `Linear` is
handed to you the way `nnx.Linear` is.

In [ ]:
# Colab setup (no-op when running locally).
# jax-judge is not published on PyPI, so the judge is installed from the
# repo itself. Regenerate with JAXCODE_REPO=you/YourFork to point this at
# your own fork:  JAXCODE_REPO=you/JAXCode make notebooks
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q flax optax')
    get_ipython().run_line_magic(
        'pip', 'install -q git+https://github.com/YOUR-GITHUB-USERNAME/JAXCode.git')
except ImportError:
    pass

In [ ]:
import jax
import jax.numpy as jnp
from flax import nnx

print("JAX", jax.__version__, "|", jax.devices())

In [ ]:
# ✏️ YOUR IMPLEMENTATION HERE

import jax
import jax.numpy as jnp


class Linear:
    """Given to you, exactly as nnx.Linear is given to you in problem 23."""

    def __init__(self, d_in, d_out, *, key):
        self.kernel = jax.random.normal(key, (d_in, d_out)) / jnp.sqrt(d_in)
        self.bias = jnp.zeros((d_out,))

    def __call__(self, x):
        return x @ self.kernel + self.bias


class MultiHeadCrossAttention:
    """Queries from x_q attend over keys/values from x_kv."""

    def __init__(self, d_model, num_heads, *, key):
        pass  # Replace this

    def __call__(self, x_q, x_kv):
        pass  # Replace this

In [ ]:
# 🔍 Scratch cell — poke at your implementation
import jax
import jax.numpy as jnp

ca = MultiHeadCrossAttention(8, 2, key=jax.random.key(0))

x_q = jax.random.normal(jax.random.key(1), (2, 3, 8))    # 3 queries
x_kv = jax.random.normal(jax.random.key(2), (2, 7, 8))   # 7 keys/values
print("seq_q=3, seq_kv=7 ->", ca(x_q, x_kv).shape)

x = jax.random.normal(jax.random.key(3), (2, 5, 8))
print("\ncross(x, x) is self-attention — the best single check of the wiring")
print("  shape:", ca(x, x).shape)

In [ ]:
# ✅ SUBMIT — run this cell to check your solution
from jax_judge import check, hint, solution, status

check("cross_attention_pure")

# hint("cross_attention_pure")      # stuck? nudge without the answer
# solution("cross_attention_pure")  # spoiler: the reference implementation
# status()                          # your dashboard across all problems